In [ ]:
#LOading the best checkpoint 
from darts.models import TFTModel

MODEL_NAME = 'daily_tft_negbin_iteration3_2026-09-18_20_10_56'
loaded_model = TFTModel.load_from_checkpoint(model_name=MODEL_NAME,best=True,map_location="cpu")
print("Model loaded successfully with all dimensions intact!")

In [ ]:
#Loading the CACHE_DIR
BASE = os.getcwd()
CACHE_DIR       = os.path.join(os.getcwd(), "series_cache")


#Loading the ROLES Json : This helps to avoid hardcoding the variables
with open(os.path.join(BASE,"column_roles.json")) as f:
    ROLES = json.load(f)

time_col,group_col,target_col = ROLES["time_col"],ROLES["group_col"],ROLES["target_col"]

FREQ = ROLES["freq"]
static_covariates = ROLES["static_covariates"]

FORECAST_START = pd.Timestamp(ROLES["forecast_start"])
FORECAST_END = pd.Timestamp(ROLES["forecast_end"])
HORIZON = (FORECAST_END - FORECAST_START).days + 1

safe_name = lambda k: str(k).replace("<>", "_").replace("/", "_").replace("\\", "_")

In [ ]:
# ---------------- SEQUENCES ----------------
class History(collections.abc.Sequence):
    """Series history to forecast from. Reads the cached 'val' split,
    which ends on the day before FORECAST_START."""

    def __init__(self, keys, statics):
        self.keys, self.statics = keys, statics

    def __len__(self):
        return len(self.keys)

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(len(self)))]
        if i < 0:
            i += len(self)
        if not 0 <= i < len(self):
            raise IndexError(i)
        with np.load(os.path.join(CACHE_DIR, f"{safe_name(self.keys[i])}.npz")) as z:
            sales, start = z["val_sales"], str(z["val_start"])
        times = pd.date_range(start, periods=len(sales), freq=FREQ)
        if times[-1] != FORECAST_START - pd.Timedelta(days=1):
            raise ValueError(f"History for {self.keys[i]} ends on {times[-1]}.")
        return TimeSeries.from_times_and_values(
            times,
            sales.reshape(-1, 1).astype(np.float32),
            columns=[target_col],
            static_covariates=self.statics[i],
        )


class SharedCov(collections.abc.Sequence):
    def __init__(self, cov, n):
        self.cov, self.n = cov, n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self.cov for _ in range(*i.indices(self.n))]
        return self.cov

In [ ]:
print(f"Forecast: {FORECAST_START.date()} -> {FORECAST_END.date()} ({HORIZON} days)")

In [ ]:
BATCH_SIZE = 512        # raise until GPU memory complains
BLOCK      = 5000       # series per block, written to disk as it goes
LIMIT      = None       # set to e.g. 2000 for a quick smoke test

In [ ]:
with open(os.path.join(CACHE_DIR, "manifest.json")) as f:
    manifest = json.load(f)

static_df = pd.read_parquet(os.path.join(CACHE_DIR, "static_covariates.parquet"))

# same dead-series filter as training, so indices stay aligned with static_df
keep = []
for k in manifest["series_keys"]:
    with np.load(os.path.join(CACHE_DIR, f"{safe_name(k)}.npz")) as z:
        keep.append(z["train_sales"].sum() > 0)

series_keys = [k for k, m in zip(manifest["series_keys"], keep) if m]
has_val     = [h for h, m in zip(manifest["has_val"],     keep) if m]
static_df   = static_df.loc[keep].reset_index(drop=True)[static_covariates].astype(str)

keys = [k for k, h in zip(series_keys, has_val) if h]
idxs = [i for i, h in enumerate(has_val) if h]
if LIMIT:
    keys, idxs = keys[:LIMIT], idxs[:LIMIT]

print(f"Series to forecast: {len(keys):,}")
if len(keys) < len(series_keys):
    print(f"  ({len(series_keys) - len(keys):,} lack enough history and are skipped)")

with open(os.path.join(CACHE_DIR, "static_cov_transformer.pkl"), "rb") as f:
    sc_transformer = pickle.load(f)

SHARED_COV = TimeSeries.from_pickle(os.path.join(CACHE_DIR, "shared_cov.pkl"))
if SHARED_COV.end_time() < FORECAST_END:
    raise ValueError(f"Covariates end {SHARED_COV.end_time().date()}, need {FORECAST_END.date()}")

# encode static covariates once, not once per series inside the loader
dummy_t = pd.date_range("2000-01-01", periods=2, freq="D")
statics = []
for i in idxs:
    t = TimeSeries.from_times_and_values(
        dummy_t, np.zeros((2, 1), dtype=np.float32), columns=[target_col],
        static_covariates=static_df.iloc[[i]].reset_index(drop=True),
    )
    statics.append(sc_transformer.transform(t).static_covariates)

# load BEST weights -- model.predict() on an in-memory model uses the LAST
# epoch, which with patience=5 is 5 epochs past what early stopping picked
model = TFTModel.load_from_checkpoint(MODEL_NAME, best=True)
print(f"Loaded best checkpoint: {MODEL_NAME}")


In [ ]:
# Every execution gets a fresh directory so old decoding results cannot be reused.
from uuid import uuid4
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
OUT_DIR = os.path.join(
    BASE, "predictions_2026_negbin_torch_convention", RUN_ID
)
print("Output directory:", OUT_DIR)

In [ ]:
# ---------------- NEGATIVE-BINOMIAL DECODING ----------------
# Darts 0.40 returns (r, p) passed to torch.distributions.NegativeBinomial.
# PyTorch mean: r*p/(1-p). SciPy nbinom requires probability 1-p.
# Do not interpret Darts' internal variable name "mu" as the reported mean.

from scipy import stats
QUANTILES = [45,55,70,75]


def summarise(p_ts):
    """Decode the predictive distribution using PyTorch's probability convention."""
    expected = [f"{target_col}_r", f"{target_col}_p"]
    if list(p_ts.components) != expected:
        raise ValueError(f"Expected {expected}, received {list(p_ts.components)}")
    v = p_ts.values(copy=False).astype(np.float64)
    r, pr = v[:, 0], v[:, 1]
    if (not np.isfinite(v).all() or np.any(r <= 0)
            or np.any((pr <= 0) | (pr >= 1))):
        raise ValueError("Invalid negative-binomial parameters; inspect precision/output.")
    # Fail on invalid parameters rather than silently clipping them.
    out = {"PRED_MEAN": r * pr / (1.0 - pr)}
    for q_int in QUANTILES:
        out[f"PRED_Q{q_int}"] = stats.nbinom.ppf(q_int / 100.0, r, 1.0 - pr)
    if not all(np.isfinite(x).all() for x in out.values()):
        raise ValueError("Non-finite decoded predictions.")
    return out


# Check the installed likelihood implementation before forecasting all series.
# Synthetic raw outputs exercise both Darts' parameter export and its training
# distribution. This check is deterministic and does not run the TFT network.
import darts
print("Darts version:", darts.__version__)
if not isinstance(model.likelihood, NegativeBinomialLikelihood):
    raise TypeError("This notebook requires NegativeBinomialLikelihood.")
if not keys or len(set(keys)) != len(keys):
    raise ValueError("Forecast series keys must be non-empty and unique.")
if not 0 < HORIZON <= model.output_chunk_length:
    raise ValueError("Parameter prediction requires horizon <= output_chunk_length.")

_raw = torch.tensor([[[[-2.0, -1.0]], [[0.4, -0.2]], [[3.0, 1.5]]]],
                    dtype=torch.float64)
_lk = model.likelihood
_dist = _lk._distr_from_params(_lk._params_from_output(_raw))
_exported = _lk.predict_likelihood_parameters(_raw).detach().cpu().numpy()
_probe_ts = TimeSeries.from_times_and_values(
    pd.date_range("2000-01-01", periods=3, freq="D"),
    _exported.reshape(3, 2), columns=[f"{target_col}_r", f"{target_col}_p"],
)
_probe_summary = summarise(_probe_ts)
np.testing.assert_allclose(
    _probe_summary["PRED_MEAN"], _dist.mean.detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
# Verify the SciPy probability conversion against Torch's probability mass.
_r, _p = _exported.reshape(3, 2).T
_counts = torch.tensor([0.0, 2.0, 5.0], dtype=torch.float64).reshape(1, 3, 1)
np.testing.assert_allclose(
    stats.nbinom.logpmf(_counts.numpy().ravel(), _r, 1.0 - _p),
    _dist.log_prob(_counts).detach().cpu().numpy().ravel(),
    rtol=1e-10, atol=1e-10,
)
print("PASS: decoded means and SciPy probabilities match the installed likelihood.")

# ---------------- FORECAST ----------------
EXPECTED_PARAMS = [f"{target_col}_r", f"{target_col}_p"]
EXPECTED_COLS   = {"PRED_MEAN", *(f"PRED_Q{q}" for q in QUANTILES)}

os.makedirs(OUT_DIR, exist_ok=True)
n_blocks = (len(keys) + BLOCK - 1) // BLOCK
paths, t0 = [], datetime.now()

for b in range(n_blocks):
    lo, hi = b * BLOCK, min((b + 1) * BLOCK, len(keys))
    path = os.path.join(OUT_DIR, f"{MODEL_NAME}_block_{b:04d}.parquet")
    paths.append(path)

    preds = model.predict(
        n=HORIZON,
        series=History(keys[lo:hi], statics[lo:hi]),
        future_covariates=SharedCov(SHARED_COV, hi - lo),
        predict_likelihood_parameters=True,   # 1 pass instead of num_samples passes
        num_samples=1,
        batch_size=BATCH_SIZE,
        verbose=False,
    )

    if len(preds) != hi - lo:
        raise ValueError("Prediction count does not match the requested series.")
    for p_ts in preds:
        if (len(p_ts) != HORIZON or p_ts.start_time() != FORECAST_START
                or p_ts.end_time() != FORECAST_END):
            raise ValueError("Forecast dates do not match the requested horizon.")

    pd.concat(
        [pd.DataFrame({group_col: k, time_col: p_ts.time_index, **summarise(p_ts)})
         for k, p_ts in zip(keys[lo:hi], preds)],
        ignore_index=True,
    ).to_parquet(path, index=False)

    el = (datetime.now() - t0).total_seconds()
    print(f"[{b+1}/{n_blocks}] {hi:,}/{len(keys):,} | {hi/el:,.0f} series/s | "
          f"ETA {(len(keys)-hi)/(hi/el)/60:.0f} min")

    del preds
    gc.collect()


# ---------------- OUTPUT ----------------
df = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
out = os.path.join(OUT_DIR, f"{MODEL_NAME}_predictions.parquet")
long_path_out = "\\\\?\\" + os.path.abspath(out)
if len(df) != len(keys) * HORIZON or df.duplicated([group_col, time_col]).any():
    raise ValueError("Unexpected row count or duplicate series/date predictions.")
df.to_parquet(long_path_out, index=False)

print(f"\nWritten -> {out}")
print(f"  rows: {len(df):,} | elapsed: {(datetime.now()-t0).total_seconds()/60:.1f} min")
for c in ["PRED_MEAN"] + [f"PRED_Q{q}" for q in QUANTILES]:
    print(f"  {c}: {df[c].sum():,.0f} ({df[c].sum()/1e5:.2f} lacs)")